In [1]:
%pip install pandas
%pip install scikit-learn
%pip install tqdm
%pip install requests
%pip install nltk
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


... (2 outputs left out)

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


  Using cached sentence_transformers-3.0.1-py3-none-any.whl (227 kB)
  Using cached torch-2.3.1-cp39-cp39-win_amd64.whl (159.7 MB)
  Using cached networkx-3.2.1-py3-none-any.whl (1.6 MB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


In [1]:
%pip install --upgrade ir_datasets
%pip install transformers -U

Requirement already up-to-date: ir_datasets in c:\users\erika\.pyenv\pyenv-win\versions\3.9.0\lib\site-packages (0.5.8)
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


In [2]:
import pandas as pd

df_rows_for_msmarcodocs = pd.read_csv('training_data/msmarco-docs-lookup.tsv', sep='\t', names=['offset_trec', 'offset_tsv'])

In [ ]:
#result = df_rows_for_msmarcodocs.loc["D1088"]
#print(result["offset_tsv"])

In [3]:
# Function to get the document using the byte offset
def get_document_by_offset(offset_tsv):
    # Step 2: Use the offset to read the line from the large TSV file
    file_path = 'training_data/msmarco/msmarco-docs.tsv'
    with open(file_path, 'r', encoding='utf-8') as file:
        file.seek(offset_tsv)
        line = file.readline()
        docid, url, title, body = line.strip().split('\t')
        document = {
            'docid': docid,
            'url': url,
            'title': title,
            'body': body
        }
        return document

In [6]:
import ir_datasets
from itertools import islice
from compute_features import PrecomputedDocumentFeatures
import nltk
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
nltk.download('wordnet')

def extract_features(group_docs, query):
    pdf = PrecomputedDocumentFeatures(group_docs, embedding_model=embedding_model)
    group_features = pdf.extract_query_features(query)
    normalized_features = pdf.normalize_features(group_features)
    return normalized_features


def prepare_msmarco_dataset(dataset, top_k=None):
    g = dataset.scoreddocs_iter()
    if top_k is None:
        documents = list(g)  # Read all documents
    else:
        documents = list(islice(g, top_k))  # Read top_k documents
    query_cache = {}
    for query in dataset.queries_iter():
        if query.query_id in query_cache:
            continue  # Skip if already cached
        for doc in documents:
            if query.query_id == doc.query_id:
                query_cache[doc.query_id] = query.text
                break  

    # Filter documents that have actual qrels
    qrel_dict = {(qrel.query_id, qrel.doc_id): qrel.relevance for qrel in dataset.qrels_iter()}
    filtered_documents = [doc for doc in documents if (doc.query_id, doc.doc_id) in qrel_dict]

    # Prepare DataFrames for features (X) and targets (y1, y2)
    # Prepare lists for features and targets
    features = []
    targets1 = []
    targets2 = []
    groups = []

    current_query_id = None
    current_group_size = 0
    group_docs = []

    for scoreddoc in filtered_documents:
        if current_query_id is None or scoreddoc.query_id != current_query_id: # if a new query_id appears, then the group is closed
            if current_group_size > 0: # filter 0 out for appending
                groups.append(current_group_size)
                query = query_cache[current_query_id]
                group_features = extract_features(group_docs, query)
                features.extend(group_features)

            # reset for a new query
            current_query_id = scoreddoc.query_id
            current_group_size = 0
            group_docs = []

        current_group_size += 1

        result = df_rows_for_msmarcodocs.loc[scoreddoc.doc_id]

        # Retrieve the document text using the offset
        document = get_document_by_offset(result["offset_tsv"])
        group_docs.append(document)

        # Add the target scores
        targets1.append(qrel_dict[(scoreddoc.query_id, scoreddoc.doc_id)])  # target1 from qrels
        targets2.append(scoreddoc.score)  # target2 from scoreddoc

    # Append the last group size
    if current_group_size > 0:
        groups.append(current_group_size)
        query = query_cache[current_query_id]
        group_features = extract_features(group_docs, query)
        features.extend(group_features)

    # Create DataFrames for features and targets
    X = pd.DataFrame(features, columns=[f"feature_{i}" for i in range(len(features[0]))])
    y = pd.DataFrame({'target1': targets1, 'target2': targets2})

    # Saving the information in three DataFrames: X, y, and groups
    X.to_csv("X.csv", index=False)
    y.to_csv("y.csv", index=False)
    pd.Series(groups, name="group").to_csv("groups.csv", index=False)

    return X, y, groups

dataset = ir_datasets.load("msmarco-document/trec-dl-2019/judged")
X, y, groups = prepare_msmarco_dataset(dataset, top_k=None) # qrels are sparsely labeled, not in the beginning; 4.3k scored docs

c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\erika\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\erika\AppD

In [25]:
# or load from already processed
import pandas as pd

X = pd.read_csv("X.csv")
y = pd.read_csv("y.csv")
groups_df = pd.read_csv("groups.csv")
groups = groups_df['group']


Combine human annotated relevance score and automatic scores.

In [26]:
import numpy as np

# Assign weights inversely proportional to their occurrence. 
# This means that less frequent (more relevant) categories get higher weights, while more frequent (less relevant) categories get lower weights.
weights = {
    0: 1,
    1: 59.4 / 28.3,
    2: 59.4 / 7.1,
    3: 59.4 / 5.2
}

# Apply weights to the relevance
y['weighted_relevance'] = y['target1'].map(weights)
y['combined_score'] = y['weighted_relevance'] + y['target2']
# Normalized
y['combined_score'] = (y['combined_score'] - y['combined_score'].min()) / (y['combined_score'].max() - y['combined_score'].min())

# Convert the normalized scores to integer ranks (e.g., 1 to 5)
y['ranked_combined_score'] = np.ceil(y['combined_score'] * 4) + 1

# Check the distribution of the ranked scores
print(y['ranked_combined_score'].value_counts())



ranked_combined_score
2.0    1412
3.0     512
5.0     348
4.0     288
1.0       1
Name: count, dtype: int64


In [27]:
X.head(10)

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19
0,1.000000,0.336493,0.092923,0.327869,1.000000,0.538122,0.074262,0.333333,0.222222,0.25,0.011514,0.950067,0.0,0.0,0.20,0.0,0.0,1.000000,0.142857,0.145038
1,0.450447,0.293839,0.129299,0.290984,0.438248,0.594648,0.458645,0.666667,0.222222,0.50,0.326996,0.944561,0.0,0.0,0.00,0.0,0.0,0.957031,0.142857,0.160305
2,0.588311,0.279621,0.054565,0.262295,0.744718,0.583446,1.000000,1.000000,0.222222,0.75,1.000000,1.000000,0.0,0.0,1.00,0.0,0.0,0.952838,0.285714,0.351145
3,0.316078,0.407583,0.317306,0.397541,0.100902,0.385051,0.074262,0.333333,0.055556,0.25,0.010678,0.811782,0.0,0.0,0.35,0.0,0.0,0.623788,0.285714,0.267176
4,0.499689,0.630332,0.579867,0.602459,0.093457,0.205739,0.074262,0.333333,0.055556,0.25,0.010678,0.702117,0.0,0.0,0.55,0.0,0.0,0.871072,0.000000,0.129771
5,0.881515,0.161137,0.050783,0.229508,0.594655,0.662140,1.000000,1.000000,0.444444,1.00,0.640116,0.865229,0.0,0.0,0.20,0.0,0.0,0.827469,0.142857,0.267176
6,0.471728,0.208531,0.092022,0.196721,0.450122,0.779528,0.458645,0.666667,0.222222,0.50,0.373833,0.981142,0.0,0.0,0.10,0.0,0.0,0.107028,0.142857,0.267176
7,0.697592,0.222749,0.379795,0.217213,0.142633,0.250252,1.000000,1.000000,0.277778,0.75,0.604470,0.793095,0.0,0.0,0.60,0.0,0.0,0.745476,0.285714,0.427481
8,0.189010,0.080569,0.014767,0.061475,0.275502,0.456860,0.074262,0.333333,0.500000,0.25,0.006860,0.902447,0.0,0.0,0.20,0.0,0.0,0.769823,0.142857,0.274809
9,0.314485,0.165877,0.126058,0.209016,0.291860,0.748846,0.458645,0.666667,0.222222,0.50,0.285466,0.886512,0.0,0.0,0.10,0.0,0.0,0.072678,0.142857,0.267176


In [28]:
y.head(10)

,target1,target2,weighted_relevance,combined_score,ranked_combined_score
0,2,-4.69200,8.366197,0.762588,5.0
1,3,-4.81849,11.423077,0.958804,5.0
2,2,-4.82038,8.366197,0.753992,5.0
3,1,-5.06599,2.098940,0.317896,3.0
4,2,-5.09501,8.366197,0.735603,4.0
5,1,-5.11022,2.098940,0.314934,3.0
6,1,-5.14148,2.098940,0.312841,3.0
7,2,-5.22266,8.366197,0.727055,4.0
8,2,-5.23613,8.366197,0.726153,4.0
9,2,-5.28076,8.366197,0.723165,4.0


Do a train-test split.

In [45]:
# Calculate the indices for the train-test split based on the groups
train_indices = sum(groups[:39])
test_indices = sum(groups[39:])

# Split the data based on the calculated indices
X_train = X[:train_indices]
X_test = X[train_indices:]
y_train = y[:train_indices]
y_test = y[train_indices:]
groups_train = groups[:39]
groups_test = groups[39:]

GBM training

In [13]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\erika\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


In [36]:
X_test

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19
2413,1.000000,0.993080,1.000000,0.733333,0.243945,0.367311,0.000000,0.000000,0.039474,0.000000,0.000000,0.793923,0.0,0.0,0.043478,0.0,0.0,0.146005,0.166667,0.120000
2414,0.019932,0.093426,0.030132,0.142222,0.016254,0.899273,0.018227,0.066667,0.013158,0.066667,0.068744,0.973377,0.0,0.0,0.347826,0.0,0.0,0.980809,0.166667,0.186667
2415,0.212306,0.095156,0.016599,0.088889,1.000000,0.452458,0.306710,0.333333,0.092105,0.133333,1.000000,0.545050,0.0,0.0,0.217391,0.0,0.0,0.599422,0.166667,0.293333
2416,0.030234,0.207612,0.064968,0.368889,0.002236,0.813586,0.018227,0.066667,0.013158,0.066667,0.072101,0.966873,0.0,0.0,0.043478,0.0,0.0,0.926852,0.000000,0.253333
2417,0.016390,0.093426,0.031136,0.097778,0.006481,0.787983,0.018227,0.066667,0.013158,0.066667,0.053733,0.885031,0.0,0.0,0.043478,0.0,0.0,0.126415,0.000000,0.226667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2556,0.100244,0.328947,0.355350,0.365079,0.088063,0.324104,0.000000,0.000000,0.272727,0.000000,0.000000,0.499792,NaN,NaN,0.571429,NaN,NaN,0.400675,0.250000,0.048193
2557,0.083172,0.421053,0.404227,0.428571,0.011998,0.433052,0.000000,0.000000,0.454545,0.000000,0.000000,0.542315,NaN,NaN,0.285714,NaN,NaN,0.487061,0.250000,0.481928
2558,0.000000,0.000000,0.007926,0.000000,0.003655,0.611660,0.083704,0.222222,0.090909,0.250000,0.216948,0.745247,NaN,NaN,0.000000,NaN,NaN,0.725682,0.250000,0.000000
2559,0.001098,0.000000,0.260238,0.015873,0.000000,0.470637,0.083704,0.222222,0.909091,0.250000,0.023548,0.534574,NaN,NaN,0.214286,NaN,NaN,0.466613,1.000000,1.000000


In [47]:
import xgboost as xgb
# Set up the parameters for LambdaMART
params = {
    'objective': 'rank:ndcg',  
    'learning_rate': 0.1,
    'gamma': 1.5,
    'min_child_weight': 0.1,
    'max_depth': 5,
    'n_estimators': 2000,
    'eval_metric': 'ndcg',
    'seed': 42
}

# Initialize the XGBRanker
ranker = xgb.XGBRanker(**params)

# Fit the model using the combined_score
#ranker.fit(X_train, y_train['ranked_combined_score'], group=groups_train)
ranker.fit(X_train, y_train['ranked_combined_score'], group=groups_train,  verbose=True)



# Predict on the test set
y_pred = ranker.predict(X_test)

# Print the evaluation results
print("Predictions on the test set:")
print(y_pred)


Predictions on the test set:
[-0.6624787   1.1698968   0.37546048  1.387013   -0.6769511  -0.87596285
 -0.9000852  -0.5687398   0.952242   -1.2685174  -1.2685174  -1.1195674
 -1.2167631  -1.2685174  -1.2685174  -1.2685174   0.7576119  -1.2685174
  1.2391262  -1.2685174  -1.2685174  -1.2685174  -0.8584192  -1.2685174
 -0.46075732 -1.2685174  -1.2685174  -1.2685174  -1.2685174  -1.2685174
 -1.2685174  -1.2685174  -0.86367565 -1.2685174  -1.2685174  -1.2685174
 -0.8584192  -1.2685174  -0.86367565  0.11619586  1.2538021  -0.76518226
  0.9421067   0.1706744   1.2193615   0.20670825  0.20511495 -0.70110816
  0.7364338   0.78546464 -0.42324615  0.20511495 -0.65320337 -0.6453178
  0.1706744   0.604331   -0.86367565 -0.78943455 -0.59096164 -0.59096164
 -0.86367565 -0.19316916  0.30241835 -0.70110816 -1.2685174  -0.14020918
 -0.86367565 -0.5351714  -0.697021   -0.3806092  -0.80298454 -0.19887595
  0.6651437  -0.6453178  -0.8135256  -0.0259911  -1.2685174  -0.5290697
 -0.5249969   0.41544068 -0.2

In [48]:
ranker.save_model('xgb_ranker_model_version2.json')


**Evaluation**

In [ ]:
import requests

# Here instead of hostname you can also use the ip adresse Displayed in NordVPN
url = 'http://your-db-host:5000/query'

N = 10000  # Change this to the number of documents you need

# Update the SQL query to fetch the first N documents
query = f'SELECT * FROM documents LIMIT {N}'

# Basic authentication details
import os
auth = ('mseproject', os.environ.get('MSE_DB_PASSWORD', ''))

# Make the POST request to the Flask API with your query
response = requests.post(url, json={'query': query}, auth=auth)

# Check if the request was successful
if response.status_code == 200:
    # Parse the JSON response
    data = response.json()
    if data:
        #here i can access the result via the index "count" because i renamed it ^
        count = data
        print(f"Total documents count: {count}")
    else:
        print("No data returned.")
else:
    print(f"Error executing query: {response.status_code} - {response.text}")

In [ ]:
print(data[0])

## Evaluate model

In [ ]:
query1 = "tübingen attractions"
query2 = "food and drinks"
query3 = "Medical faculty tübingen"
query3 = "best hospitals tübingen"
query4 = "tübingen AI campus"
query5 = "Clinical trials in Tübingen"
query6 = "Public transportation in Tübingen"
query7 = "Parks and recreation areas in Tübingen"
query8 = "Healthcare facilities in Tübingen"
query9 = "Universität Tübingen campus tours"

In [ ]:
from bm25 import BM25S
from compute_features import tokenize
from multiprocessing import cpu_count

corpus =  []
for i in data:
    corpus.append(tokenize(i["content"]))

bm25 = BM25S(corpus, num_threads=cpu_count())

In [ ]:
query = tokenize(query1)
scores = bm25.get_scores(query)

# Get the 50 highest scores
top_50_scores_indices = scores.argsort()[-50:][::-1]
top_50_scores = scores[top_50_scores_indices]



In [ ]:
documents = []

# Printing top 50 scores and corresponding document indices
for index in top_50_scores_indices:
    #print(f"Document Index: {index}, Score: {scores[index]}")
    documents.append(data[index])

In [ ]:
from compute_features import PrecomputedDocumentFeatures
import nltk

nltk.download('wordnet')

def extract_features(group_docs, query):
    pdf = PrecomputedDocumentFeatures(group_docs)
    group_features = pdf.extract_query_features(query)
    normalized_features = pdf.normalize_features(group_features)
    return normalized_features

In [ ]:
# Rename "content" key to "body"
for doc in documents:
    doc["body"] = doc.pop("content")


In [ ]:
features = extract_features(documents, query1)

In [ ]:
import xgboost as xgb
import numpy as np

# Initialize a new XGBRanker
ranker = xgb.XGBRanker()

# Load the model from the file
ranker.load_model('xgb_ranker_model.json')
# Predict on the test set
y_pred = ranker.predict(features)

# Print the evaluation results
print("Predictions on the test set:")

# Define the number of top documents you want to retrieve
n = 10

# Get the indices of the top n documents based on the predicted scores
top_n_indices = np.argsort(y_pred)[-n:][::-1]

# Retrieve the top n documents
top_n_documents = [documents[i] for i in top_n_indices]

# Print the top n documents
print("Top n documents:")
for doc in top_n_documents:
    print(f"URL: {doc['url']}\nBody: {doc['body']}\n")


****From the past: Test on Microsoft Dataset****

In [ ]:
import numpy as np
from collections import Counter
from typing import List, Dict
import pandas as pd
from bm25 import BM25S

class PrecomputedDocumentFeatures:
    
    def __init__(self, documents: List[dict]):
        self.documents = documents
        self.doc_count = len(documents)
        self.idf_cache = {}
        self.precomputed_features = []
        self.titles = []
        self.bodies = []
        self.urls = []
        self.whole_documents = []
        self._build_idf_cache()
        self._precompute_features()
        
    
    def _build_idf_cache(self): 
        """ 
        Pre-compute IDF for all terms in the corpus. 
        WARNING: IDF defined as: 1 divided by the number of documents containing the query terms.
        """
        term_doc_count = Counter()
        for doc in self.documents:
            content = doc['content']
            url = doc['url']
            outgoing_links = doc['outgoing_links']
            whole_document = f"{content} {url} {outgoing_links}".lower() # lower!
            unique_terms = set(whole_document.split())
            for term in unique_terms:
                term_doc_count[term] += 1
        
        for term, count in term_doc_count.items():
            self.idf_cache[term] = 1 / (count + 1)
    
    def compute_idf(self, term: str) -> float:
        """ Compute IDF for a given term using the pre-computed IDF cache. """
        return self.idf_cache.get(term, 1 / (1 + 1))  # Default IDF for unseen terms
    
    def stream_length(self, text: str) -> int:
        """ Determine the length of the text in terms of word count. """
        return len(text.split())
    
    def term_frequencies(self, text: str) -> Dict[str, int]:
        """ Compute term frequencies for a given text. """
        return Counter(text.split())
    
    def _trim_content(self, content: str) -> str:
        """ Trim 20% off the beginning and end of the content as body. """
        words = content.split()
        total_length = len(words)
        cut_off = int(total_length * 0.2)
        trimmed_words = words[cut_off: total_length - cut_off]
        return ' '.join(trimmed_words)
    
    def _precompute_features(self):
        """ Pre-compute features for all documents. """
        for doc in self.documents:
            content = doc['content'].lower()
            body = self._trim_content(content)
            title = doc['title'].lower()
            url = doc['url'].lower()
            outgoing_links = doc['outgoing_links'].lower()
            url_splitted = url.replace('/', ' ').replace('.', ' ').split()
            url_terms = ' '.join(url_splitted)
            outgoing_links_terms = ' '.join(outgoing_links.replace('/', ' ').replace('.', ' ').split())
            whole_document = f"{content} {url_terms} {outgoing_links_terms}"
            
            body_term_freq = self.term_frequencies(body)
            title_term_freq = self.term_frequencies(title)
            url_term_freq = self.term_frequencies(url_terms)
            whole_doc_term_freq = self.term_frequencies(whole_document)
            
            body_features = {
                'stream_length': self.stream_length(body),
                'term_freq': body_term_freq
            }
            
            title_features = {
                'stream_length': self.stream_length(title),
                'term_freq': title_term_freq
            }
            
            url_features = {
                'stream_length': len(url_splitted[2]), # only the main part of the url e.g. tuebingen from https://www.tuebingen.de
                'term_freq': url_term_freq
            }
            
            whole_doc_features = {
                'stream_length': self.stream_length(whole_document),
                'term_freq': whole_doc_term_freq
            }
            
            # For each document: precomputed features
            self.precomputed_features.append({
                'content': body_features,
                'title': title_features,
                'url': url_features,
                'whole_document': whole_doc_features,
                'num_slashes_url': url.count('/'),
                'num_char_url': len(url),
                'outlink_number': len(doc['outgoing_links'].split(",")),
                'pagerank': doc['pagerank'] * 100000 # too low to be in the MS dataset
            })

            self.titles.append(title.split())
            self.urls.append(url_splitted)
            self.whole_documents.append(whole_document.split())
            self.bodies.append(body.strip())
    
    def extract_query_features(self, query: str, order=False, bm25=False) -> pd.DataFrame:
        """ Extract query-specific features using pre-computed document features. """
        query_terms = query.lower().split()
        all_features = []
        
        part_order = {'content': 1, 'title': 3, 'url': 4, 'whole_document': 5}

        for doc_features in self.precomputed_features: # for each document
            features = {}
            for part in ['content', 'title', 'url', 'whole_document']:
                term_freq = doc_features[part]['term_freq']
                # Use the precomputed IDF values for the query terms
                idf_values = [self.compute_idf(term) for term in query_terms]
                
                query_term_freqs = [term_freq[term] for term in query_terms if term in term_freq]
                tf_idf_values = [term_freq[term] * idf for term, idf in zip(query_terms, idf_values) if term in term_freq]
                
                covered_query_terms = len(query_term_freqs)
                sum_term_freq = sum(query_term_freqs)
                min_term_freq = min(query_term_freqs, default=0)
                max_term_freq = max(query_term_freqs, default=0)
                mean_term_freq = round(sum(query_term_freqs) / len(query_term_freqs), 6) if query_term_freqs else 0.000000
                var_term_freq = round(sum((x - mean_term_freq) ** 2 for x in query_term_freqs) / len(query_term_freqs), 6) if query_term_freqs else 0.000000
                
                sum_tf_idf = round(sum(tf_idf_values), 6) if tf_idf_values else 0.000000
                min_tf_idf = round(min(tf_idf_values), 6) if tf_idf_values else 0.000000
                max_tf_idf = round(max(tf_idf_values), 6) if tf_idf_values else 0.000000
                mean_tf_idf = round(sum(tf_idf_values) / len(tf_idf_values), 6) if tf_idf_values else 0.000000
                var_tf_idf = round(sum((x - mean_tf_idf) ** 2 for x in tf_idf_values) / len(tf_idf_values), 6) if tf_idf_values else 0.000000
                
                # IDF feature for the query terms
                idf = round(sum(idf_values) / len(idf_values), 6) if idf_values else 0.000000
                
                feature_values = [
                    covered_query_terms,
                    covered_query_terms / len(query_terms),
                    doc_features[part]['stream_length'],
                    idf,
                    sum_term_freq,
                    min_term_freq,
                    max_term_freq,
                    mean_term_freq,
                    var_term_freq,
                    sum_tf_idf,
                    min_tf_idf,
                    max_tf_idf,
                    mean_tf_idf,
                    var_tf_idf,
                    doc_features['num_slashes_url'],
                    doc_features['num_char_url'],
                    doc_features['outlink_number'],
                    doc_features['pagerank']
                ]
                
                # Microsoft Dataset feature names
                feature_names = [
                    part_order[part],
                    part_order[part] + 5,
                    part_order[part] + 10,
                    part_order[part] + 15,
                    part_order[part] + 20,
                    part_order[part] + 25,
                    part_order[part] + 30,
                    part_order[part] + 35,
                    part_order[part] + 40,
                    part_order[part] + 70,
                    part_order[part] + 75,
                    part_order[part] + 80,
                    part_order[part] + 85,
                    part_order[part] + 90,
                    126,
                    127,
                    129,
                    130
                ]
                for name, value in zip(feature_names, feature_values):
                    features[name] = value
            all_features.append(features)
        
        if bm25:
            from multiprocessing import cpu_count
            cpu_count = cpu_count()
            query_split = query.split()
            bm25 = BM25S(self.titles, num_threads=cpu_count, method="bm25+")
            scores = bm25.get_scores(query_split)
            for i, features in enumerate(all_features):
                features[108] = scores[i]

            bm25 = BM25S(self.bodies, num_threads=cpu_count, method="bm25+")
            scores = bm25.get_scores(query_split)
            for i, features in enumerate(all_features):
                features[106] = scores[i]

            bm25 = BM25S(self.urls, num_threads=cpu_count, method="bm25+")
            scores = bm25.get_scores(query_split)
            for i, features in enumerate(all_features):
                features[109] = scores[i]

                
            bm25 = BM25S(self.whole_documents, num_threads=cpu_count, method="bm25+")
            scores = bm25.get_scores(query_split)
            for i, features in enumerate(all_features):
                features[110] = scores[i]

        df_results = pd.DataFrame(all_features)

        if order:
            df_results = df_results.reindex(sorted(df_results.columns, key=int), axis=1)
        
        return df_results
# Example usage
documents = count

query = "Hölderlin Tower on the Neckar"
import time 

start = time.time()
feature_extractor = PrecomputedDocumentFeatures(documents) # assume the documents are not lowercased, not NLP perf for now
query_features = feature_extractor.extract_query_features(query, order=True, bm25=True)
stop = time.time()
times = stop - start
query_features


In [ ]:
query_features[66]

In [ ]:

import pandas as pd
import numpy as np

def get_data(data_path, top_k = None):
    if top_k:
        dfs = {
            "train": pd.read_csv(f"{data_path}/train.txt", delimiter=" ", header=None).head(top_k),
            "vali": pd.read_csv(f"{data_path}/vali.txt", delimiter=" ", header=None).head(top_k),
            "test": pd.read_csv(f"{data_path}/test.txt", delimiter=" ", header=None).head(top_k),
        }
    else:
        dfs = {
            "train": pd.read_csv(f"{data_path}/train.txt", delimiter=" ", header=None),
            "vali": pd.read_csv(f"{data_path}/vali.txt", delimiter=" ", header=None),
            "test": pd.read_csv(f"{data_path}/test.txt", delimiter=" ", header=None),
        }


    for df in dfs.values():
        df.columns = np.arange(len(df.columns))
        df.drop(columns=df.columns[df.isna().all()].tolist(), inplace=True)  # delete columns where all data is missing

    split = {}
    split["X_train"] = dfs["train"].iloc[:, 1:]
    split["X_val"] = dfs["vali"].iloc[:, 1:]
    split["X_test"] = dfs["test"].iloc[:, 1:]

    y_train = dfs["train"].iloc[:, 0]
    y_val = dfs["vali"].iloc[:, 0]
    y_test = dfs["test"].iloc[:, 0]

    # Create group parameters for LightGBM
    group_train = dfs["train"].groupby(1).size().to_list()
    group_vali = dfs["vali"].groupby(1).size().to_list()
    group_test = dfs["test"].groupby(1).size().to_list()

    # Specify the columns to keep
    columns_to_keep = [1, 3, 4, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 18, 19, 20, 21, 23, 24, 25, 26, 28, 29, 30, 31, 33, 34, 35, 36, 38, 39, 40, 41, 43, 44, 45, 71, 73, 74, 75, 76, 78, 79, 80, 81, 83, 84, 85, 86, 88, 89, 90, 91, 93, 94, 95, 106, 108, 109, 110, 126, 127, 129, 130] 
    # add ti

    for name, df in split.items():
        df = df.applymap(lambda x: x.split(":", 1)[-1])  # Get rid of irrelevant information
        df = df.astype(float)  # Convert data into float format
        df = df.drop(columns=1)  # Drop the query ID column
        df.columns = [i for i in range(1, 137)]  # Rename column indices
        df = df[columns_to_keep]  # Keep only specified columns
        split[name] = df

    return (
        split["X_train"],
        split["X_test"],
        split["X_val"],
        y_train,
        y_test,
        y_val,
        group_train,
        group_vali,
        group_test
    )

In [ ]:
# Load data
data_path = "training_data/Fold1"
X_train, X_test, X_val, y_train, y_test, y_val, group_train, group_vali, group_test = get_data(data_path)

In [ ]:
X_val[130]

In [ ]:
from multiprocessing import cpu_count
import lightgbm as lgb
import tqdm
cpu_count = cpu_count()
print(cpu_count)
total_boost_round = 1000 # over 1000 its just repeating

# Train the LightGBM model
lgb_ranker = lgb.LGBMRanker(
    objective='lambdarank',
    boosting_type='dart', # enhancement of gradient boosting decision tree
    num_leaves=31,
    learning_rate=0.05,
    n_estimators=total_boost_round,  # Maximum number of boosting rounds; epochs
    n_jobs=cpu_count
    
)

lgb_ranker.fit(
    X_train, y_train,
    group=group_train,
    eval_set=[(X_test, y_test)],
    eval_group=[group_test],
    eval_at=[10],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50), # early stopping not available with dart,
        lgb.log_evaluation(period=1, show_stdv=True),  # Log evaluation results
    ]
)

In [ ]:
# Predict relevance scores for the test set
y_pred = lgb_ranker.predict(query_features, pred_contrib=False)

In [ ]:
import pandas as pd
# Convert SHAP values to a DataFrame for easier interpretation
shap_df = pd.read_csv("shap.csv")

# Print the first few rows of the SHAP values DataFrame
print(shap_df.head())

In [ ]:
import shap
shap_values_no_bias = shap_df.drop(columns=["bias", "Unnamed: 0"])
shap_values = shap_values_no_bias.values

# Summary plot
shap.summary_plot(shap_values, feature_names=shap_values_no_bias.columns)

In [ ]:
%pip install shap
%pip install matplotlib

In [ ]:
# Save the model to a file
model_filename = 'lgb_ranker_model1.txt'
lgb_ranker.booster_.save_model(model_filename)

**Test on our data**

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMRanker
# Load the model from the file
ranker = lgb.Booster(model_file='lgb_ranker_model1.txt')


In [ ]:
predictions = ranker.predict(query_features)

In [ ]:
predictions.shape

In [ ]:
# Get the indices of the top 10 predictions
top_10_indices = np.argsort(predictions)[-10:][::-1]
print("Indices of the top 10 predictions:", top_10_indices)

for i in top_10_indices:
    print(count[i])

In [ ]:
count[45]